# QM 640 Capstone — Step 6: RQ1-RQ4 Statistical Tests

Runs the exact tests from Table 1 of the Synopsis, each preceded by a
Shapiro-Wilk normality check with the specified non-parametric fallback.

- **RQ1:** One-sample t-test on CAR_short (H0: CAR = 0) — fallback: Wilcoxon signed-rank
- **RQ2:** One-way ANOVA across announcement_type — fallback: Kruskal-Wallis; Tukey HSD post-hoc if significant
- **RQ3:** Multiple regression with interaction (firm_size x announcement_type)
- **RQ4:** Independent-samples t-test, tech vs. non-tech — Levene's test for variance equality; Welch's correction if violated

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 1126, done.
remote: Counting objects: 100% (313/313), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 1126 (delta 125), reused 251 (delta 92), pack-reused 813 (from 1)
Receiving objects: 100% (1126/1126), 8.24 MiB | 19.01 MiB/s, done.
Resolving deltas: 100% (566/566), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas numpy scipy statsmodels scikit-learn

## Cell 3 — Configuration

In [3]:
import os

DATA_FILE = os.path.join(BASE_DIR, "data/processed/analysis_dataset.csv")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ALPHA = 0.05

import pandas as pd
df = pd.read_csv(DATA_FILE)
print(f"Loaded {len(df)} events from {DATA_FILE}")
df.head()

Loaded 445 events from /content/QM640-WALSH-CAPSTONE/data/processed/analysis_dataset.csv


,event_id,ticker,event_date,announcement_type,firm_size_log,firm_size_source,sector,sector_source,alpha,beta,CAR_short,CAR_long,sector_binary,log_assets,leverage_ratio,rd_intensity,prior_ai_disclosure_count
0,0001193125-25-155007:d91487dex991.htm,XRX,2025-07-02,M&A,19.774366,Direct gap-fill (yfinance),Industrials,Direct gap-fill (yfinance),-0.002471,1.363147,-0.042240,-0.240312,Non-Technology,22.906391,0.847757,NaN,0
1,0001213900-26-069795:ea029521601ex99-1.htm,MYSE,2026-06-18,partnership,16.210170,Direct gap-fill (yfinance),Technology,Direct gap-fill (yfinance),0.003747,2.119101,0.224378,-0.110368,Technology,15.413890,0.218199,150.684932,5
2,0001493152-26-008435:ex99-1.htm,KAPA,2026-03-02,M&A,15.695581,Direct gap-fill (yfinance),Healthcare,Direct gap-fill (yfinance),-0.000544,2.442795,0.093387,-0.019670,Non-Technology,15.693448,0.151682,NaN,1
3,0001185185-25-000690:mitiex99-1.htm,MITI,2025-06-25,R&D,13.638323,Direct gap-fill (yfinance),Technology,Direct gap-fill (yfinance),0.007699,0.467415,0.079106,-0.746096,Technology,12.152598,117.841671,0.310706,0
4,0001213900-25-059641:ea024758501ex99-1_richtec...,RR,2025-06-30,M&A,19.576008,Direct gap-fill (yfinance),Industrials,Direct gap-fill (yfinance),0.019502,2.184100,-0.078662,-0.555883,Non-Technology,18.491409,0.012839,0.371286,1


In [4]:
# --- Documented event exclusions (applied before RQ1-RQ4) ---
df_baseline = df.copy()  # keep full, unfiltered sample for the sensitivity check below

EXCLUDED_TICKERS = {
    "BSAI": ("Degenerate market-model beta (1174.7) vs. sample range of "
             "roughly -7 to +9; illiquid OTC micro-cap whose estimation-window "
             "price spans multiple orders of magnitude ($0.0003 to $7.75). "
             "CAR_short = -2631%% is not economically plausible; excluded as "
             "a data-quality issue, not a genuine market reaction."),
}

HOLDOUT_TICKERS = {
    "RKTO": ("Ticker symbol did not begin trading until 2026-05-28 "
             "(company traded as HOTH before then); event date is 2026-05-21. "
             "Held out pending verification that price/return data resolves "
             "to the correct pre-rename ticker."),
}

exclusion_log = []
for ticker, reason in {**EXCLUDED_TICKERS, **HOLDOUT_TICKERS}.items():
    for _, row in df[df["ticker"] == ticker].iterrows():
        exclusion_log.append({
            "event_id": row["event_id"], "ticker": ticker,
            "event_date": row["event_date"], "reason": reason,
            "status": "excluded" if ticker in EXCLUDED_TICKERS else "holdout",
        })

n_before = len(df)
df = df[~df["ticker"].isin(set(EXCLUDED_TICKERS) | set(HOLDOUT_TICKERS))].copy()

pd.DataFrame(exclusion_log).to_csv(
    os.path.join(RESULTS_DIR, "event_exclusion_log.csv"), index=False
)

print(f"n before exclusions: {n_before}")
print(f"n after exclusions:  {len(df)}  (excluded/held out: {n_before - len(df)})")
print(f"BNAI and MYSE retained deliberately — real market reactions, "
      f"not data errors; see conversation/report notes.")

n before exclusions: 445
n after exclusions:  442  (excluded/held out: 3)
BNAI and MYSE retained deliberately — real market reactions, not data errors; see conversation/report notes.


## RQ1 — Does CAR differ significantly from zero?

In [5]:
from scipy import stats

def rq1_test(df):
    car = df["CAR_short"].dropna()
    _, p_norm = stats.shapiro(car)
    print(f"Shapiro-Wilk normality test: p = {p_norm:.4f}")

    if p_norm >= ALPHA:
        t_stat, p_val = stats.ttest_1samp(car, 0)
        print(f"One-sample t-test: t = {t_stat:.3f}, p = {p_val:.4f}")
        method = "one-sample t-test"
    else:
        stat, p_val = stats.wilcoxon(car)
        t_stat = stat
        print(f"Normality violated -> Wilcoxon signed-rank test: W = {stat:.3f}, p = {p_val:.4f}")
        method = "Wilcoxon signed-rank (non-parametric fallback)"

    caar = car.mean()
    print(f"CAAR (mean CAR): {caar:.4%}")
    print(f"Result: {'REJECT' if p_val < ALPHA else 'FAIL TO REJECT'} H0 at alpha = .05")

    return {"RQ": "RQ1", "method": method, "statistic": t_stat, "p_value": p_val,
            "n": len(car), "CAAR": caar}


rq1_result = rq1_test(df)

Shapiro-Wilk normality test: p = 0.0000
Normality violated -> Wilcoxon signed-rank test: W = 40845.000, p = 0.0026
CAAR (mean CAR): 0.4018%
Result: REJECT H0 at alpha = .05


## RQ2 — Does CAR differ by announcement type?

In [6]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def rq2_test(df):
    groups = [g["CAR_short"].dropna() for _, g in df.groupby("announcement_type")]

    normal = all(stats.shapiro(g)[1] >= ALPHA for g in groups if len(g) >= 3)
    _, p_levene = stats.levene(*groups)
    print(f"Normality (all groups): {'OK' if normal else 'violated'} | "
          f"Levene's test (equal variance): p = {p_levene:.4f}")

    if normal:
        f_stat, p_val = stats.f_oneway(*groups)
        print(f"One-way ANOVA: F = {f_stat:.3f}, p = {p_val:.4f}")
        method = "one-way ANOVA"
        if p_val < ALPHA:
            tukey = pairwise_tukeyhsd(df["CAR_short"].dropna(),
                                       df.loc[df["CAR_short"].notna(), "announcement_type"])
            print("\nTukey HSD post-hoc comparisons:")
            print(tukey)
    else:
        h_stat, p_val = stats.kruskal(*groups)
        f_stat = h_stat
        print(f"Normality violated -> Kruskal-Wallis: H = {h_stat:.3f}, p = {p_val:.4f}")
        method = "Kruskal-Wallis (non-parametric fallback)"

    ss_between = sum(len(g) * (g.mean() - df["CAR_short"].mean()) ** 2 for g in groups)
    ss_total = ((df["CAR_short"].dropna() - df["CAR_short"].mean()) ** 2).sum()
    eta_sq = ss_between / ss_total if ss_total else float("nan")
    print(f"Effect size (eta-squared): {eta_sq:.4f}")

    return {"RQ": "RQ2", "method": method, "statistic": f_stat, "p_value": p_val,
            "n": len(df), "eta_squared": eta_sq}


rq2_result = rq2_test(df)

Normality (all groups): violated | Levene's test (equal variance): p = 0.0821
Normality violated -> Kruskal-Wallis: H = 1.300, p = 0.5220
Effect size (eta-squared): 0.0034


## RQ3 — Does firm size moderate announcement-type -> CAR?

In [7]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

def rq3_test(df):
    model_df = df.dropna(subset=["CAR_short", "firm_size_log", "announcement_type"]).copy()
    model = smf.ols("CAR_short ~ firm_size_log * C(announcement_type)", data=model_df).fit()
    print(model.summary())

    X = sm.add_constant(pd.get_dummies(
        model_df[["firm_size_log", "announcement_type"]], drop_first=True
    ).astype(float))
    vif = pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    })
    print("\nVariance Inflation Factors:")
    print(vif)

    return {"RQ": "RQ3", "method": "multiple regression with interaction",
            "statistic": model.fvalue, "p_value": model.f_pvalue,
            "n": len(model_df), "r_squared": model.rsquared,
            "adj_r_squared": model.rsquared_adj}


rq3_result = rq3_test(df)

                            OLS Regression Results                            
Dep. Variable:              CAR_short   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                    0.8602
Date:                Sun, 02 Aug 2026   Prob (F-statistic):              0.508
Time:                        06:08:07   Log-Likelihood:                -16.964
No. Observations:                 441   AIC:                             45.93
Df Residuals:                     435   BIC:                             70.46
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------

## RQ4 — Does CAR differ between tech and non-tech sectors?

In [8]:
def rq4_test(df):
    tech = df[df["sector_binary"] == "Technology"]["CAR_short"].dropna()
    nontech = df[df["sector_binary"] == "Non-Technology"]["CAR_short"].dropna()

    _, p_levene = stats.levene(tech, nontech)
    equal_var = p_levene >= ALPHA
    variance_note = "equal variance assumed" if equal_var else "Welch's correction applied"
    print(f"Levene's test (equal variance): p = {p_levene:.4f} -> {variance_note}")

    t_stat, p_val = stats.ttest_ind(tech, nontech, equal_var=equal_var)
    mean_diff = tech.mean() - nontech.mean()
    print(f"Independent-samples t-test: t = {t_stat:.3f}, p = {p_val:.4f}")
    print(f"Mean CAR difference (tech - non-tech): {mean_diff:.4%}")
    print(f"Result: {'REJECT' if p_val < ALPHA else 'FAIL TO REJECT'} H0 at alpha = .05")

    return {"RQ": "RQ4", "method": "independent-samples t-test", "statistic": t_stat,
            "p_value": p_val, "n": len(tech) + len(nontech), "mean_diff": mean_diff}


rq4_result = rq4_test(df)

Levene's test (equal variance): p = 0.2633 -> equal variance assumed
Independent-samples t-test: t = 0.148, p = 0.8823
Mean CAR difference (tech - non-tech): 0.3591%
Result: FAIL TO REJECT H0 at alpha = .05


## Save all results

In [9]:
results = [rq1_result, rq2_result, rq3_result, rq4_result]
results_df = pd.DataFrame(results)
results_df["dataset_version"] = "primary_excl_BSAI_holdout_RKTO"

# Sensitivity check: rerun on the untouched baseline sample
baseline_results = [rq1_test(df_baseline), rq2_test(df_baseline),
                     rq3_test(df_baseline), rq4_test(df_baseline)]
baseline_df = pd.DataFrame(baseline_results)
baseline_df["dataset_version"] = "baseline_full_445"

results_df = pd.concat([results_df, baseline_df], ignore_index=True)
results_df.to_csv(os.path.join(RESULTS_DIR, "rq_results_summary.csv"), index=False)
print(f"All results saved -> {RESULTS_DIR}/rq_results_summary.csv")
results_df

Shapiro-Wilk normality test: p = 0.0000
Normality violated -> Wilcoxon signed-rank test: W = 41386.000, p = 0.0024
CAAR (mean CAR): -5.2971%
Result: REJECT H0 at alpha = .05
Normality (all groups): violated | Levene's test (equal variance): p = 0.6053
Normality violated -> Kruskal-Wallis: H = 1.081, p = 0.5826
Effect size (eta-squared): 0.0031
                            OLS Regression Results                            
Dep. Variable:              CAR_short   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                 -0.006
Method:                 Least Squares   F-statistic:                    0.4310
Date:                Sun, 02 Aug 2026   Prob (F-statistic):              0.827
Time:                        06:08:07   Log-Likelihood:                -736.37
No. Observations:                 444   AIC:                             1485.
Df Residuals:                     438   BIC:                             1509.
Df Model:             

,RQ,method,statistic,p_value,n,CAAR,eta_squared,r_squared,adj_r_squared,mean_diff,dataset_version
0,RQ1,Wilcoxon signed-rank (non-parametric fallback),40845.000000,0.002554,442,0.004018,NaN,NaN,NaN,NaN,primary_excl_BSAI_holdout_RKTO
1,RQ2,Kruskal-Wallis (non-parametric fallback),1.300159,0.522004,442,NaN,0.003420,NaN,NaN,NaN,primary_excl_BSAI_holdout_RKTO
2,RQ3,multiple regression with interaction,0.860164,0.507850,441,NaN,NaN,0.009790,-0.001592,NaN,primary_excl_BSAI_holdout_RKTO
3,RQ4,independent-samples t-test,0.148088,0.882341,441,NaN,NaN,NaN,NaN,0.003591,primary_excl_BSAI_holdout_RKTO
4,RQ1,Wilcoxon signed-rank (non-parametric fallback),41386.000000,0.002426,445,-0.052971,NaN,NaN,NaN,NaN,baseline_full_445
5,RQ2,Kruskal-Wallis (non-parametric fallback),1.080568,0.582583,445,NaN,0.003075,NaN,NaN,NaN,baseline_full_445
6,RQ3,multiple regression with interaction,0.431041,0.826984,444,NaN,NaN,0.004896,-0.006463,NaN,baseline_full_445
7,RQ4,independent-samples t-test,0.955362,0.339917,444,NaN,NaN,NaN,NaN,0.116222,baseline_full_445


## Commit and push results back to GitHub

In [10]:
!git -C {BASE_DIR} add "results/rq_results_summary.csv"
!git -C {BASE_DIR} commit -m "Step 6: RQ1-RQ4 statistical test results"
!git -C {BASE_DIR} push

[main 3b8cb51] Step 6: RQ1-RQ4 statistical test results
 1 file changed, 9 insertions(+), 5 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 925 bytes | 925.00 KiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   78870aa..3b8cb51  main -> main
